# 1. Cài đặt và import thư viện


In [1]:
pip install datasets pandas ipykernel -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 76.6 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset
import pandas as pd
import json
import re
from pathlib import Path

# 2. Load dataset từ Hugging Face

In [4]:
dataset = load_dataset("NevenaD/MedNurse-QA", split="train")

print("Total rows:", len(dataset))
print("Columns:", dataset.column_names)

README.md:   0%|          | 0.00/6.82k [00:00<?, ?B/s]

MedNurse-QA.parquet:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21643 [00:00<?, ? examples/s]

Total rows: 21643
Columns: ['question', 'answer', 'sub-chapter', 'chapter ', 'book']


# 3. Kiểm tra dữ liệu ban đầu


In [5]:
df = dataset.to_pandas()
df.head()

,question,answer,sub-chapter,chapter,book
0,What is effective communication?,"Effective communication requires a sender, a c...",basic communication concepts,communication,nursing fundamentals
1,What is the role of nurses in communication?,Nurses help patients and their families unders...,basic communication concepts,communication,nursing fundamentals
2,What is verbal communication?,Verbal communication is an exchange of informa...,basic communication concepts,communication,nursing fundamentals
3,What is the impact of medical jargon on commun...,Using medical jargon or slang may create an un...,basic communication concepts,communication,nursing fundamentals
4,Why is it important to assess the receiver's p...,It is important to adapt communication to meet...,basic communication concepts,communication,nursing fundamentals


In [6]:
df.isnull().sum()

,0
question,0
answer,0
sub-chapter,0
chapter,0
book,0


# 4. Xây dựng hàm clean text


In [7]:
def clean_text(text):

    if text is None:
        return ""

    text = str(text)
    text = text.replace("\u00a0", " ")
    text = text.replace("\t", " ")
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")

    # Gộp nhiều khoảng trắng thành 1 khoảng trắng
    text = re.sub(r"\s+", " ", text)

    return text.strip()

# 5. Clean data và chuẩn hóa


In [12]:
cleaned_data = []
seen = set()

for row in dataset:
    question = clean_text(row.get("question", ""))
    answer = clean_text(row.get("answer", ""))

    # Bỏ dòng thiếu question hoặc answer
    if not question or not answer:
        continue

    # Bỏ dòng quá ngắn
    if len(question) < 5 or len(answer) < 5:
        continue

    # Xóa trùng lặp theo question + answer
    duplicate_key = (question.lower(), answer.lower())
    if duplicate_key in seen:
        continue

    seen.add(duplicate_key)

    # Tạo source chi tiết
    book = clean_text(row.get("book", ""))
    chapter = clean_text(row.get("chapter", ""))
    sub_chapter = clean_text(row.get("sub-chapter", ""))

    source_parts = [
        "NevenaD/MedNurse-QA",
        book,
        chapter,
        sub_chapter
    ]

    source = "/".join([part for part in source_parts if part])

    item = {
        "question": question,
        "answer": answer,
        "source": source
    }

    cleaned_data.append(item)

print("Original rows:", len(dataset))
print("Cleaned rows:", len(cleaned_data))
print("Removed rows:", len(dataset) - len(cleaned_data))

Original rows: 21643
Cleaned rows: 20765
Removed rows: 878


In [13]:
cleaned_data[:5]

[{'question': 'What is effective communication?',
  'answer': 'Effective communication requires a sender, a clear message, and a receiver who can decode and interpret the message.',
  'source': 'NevenaD/MedNurse-QA/nursing fundamentals/basic communication concepts'},
 {'question': 'What is the role of nurses in communication?',
  'answer': 'Nurses help patients and their families understand healthcare needs and treatments using verbal, nonverbal, and written communication.',
  'source': 'NevenaD/MedNurse-QA/nursing fundamentals/basic communication concepts'},
 {'question': 'What is verbal communication?',
  'answer': 'Verbal communication is an exchange of information using words understood by the receiver in a way that conveys professional caring and respect.',
  'source': 'NevenaD/MedNurse-QA/nursing fundamentals/basic communication concepts'},
 {'question': 'What is the impact of medical jargon on communication?',
  'answer': 'Using medical jargon or slang may create an unintended b

# 6. Lưu dữ liệu thành file JSON

In [ ]:
output_path = Path("mednurse_qa_cleaned_standardized.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=2)

print("Saved to:", output_path)
print("Total records:", len(cleaned_data))

Saved to: mednurse_qa_cleaned_standardized.json
Total records: 20765


In [11]:
with open(output_path, "r", encoding="utf-8") as f:
    check_data = json.load(f)

print("Loaded records:", len(check_data))
check_data[0]

Loaded records: 20765


{'question': 'What is effective communication?',
 'answer': 'Effective communication requires a sender, a clear message, and a receiver who can decode and interpret the message.',
 'source': 'NevenaD/MedNurse-QA'}

In [ ]:
# csv_path = Path("mednurse_qa_cleaned_standardized.csv")
# pd.DataFrame(cleaned_data).to_csv(csv_path, index=False, encoding="utf-8-sig")

# print("Saved CSV to:", csv_path)